# Étape 1 (fin) : un jeu de triage

Aucun des quatre corpus imposés ne donne de niveau de priorité. Le jeu SFT apprend des connaissances médicales et un format de réponse, mais rien n'apprend au modèle à classer un patient en urgence maximale, modérée ou différée.

On fabrique donc des exemples de triage à partir des cas cliniques de MediQAl déjà présents dans le jeu (ils sont donc déjà anonymisés par le notebook 02) :
- le niveau est attribué par un grand modèle, Claude Opus 5.5 (API Anthropic), qui suit l'échelle de tri FRENCH de la SFMU
- les six niveaux de FRENCH sont regroupés dans les trois niveaux de la mission
- ce sont des données synthétiques, pas des niveaux validés par des médecins : une partie est relue à la main

Sortie : `triage_train.jsonl` (400 cas) et `triage_test.jsonl` (100 cas) dans `data/processed/`, version 1.1 du jeu de données.

In [1]:
import json
import random
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import anthropic
import pandas as pd
import pymupdf
from dotenv import load_dotenv
from pydantic import BaseModel

SEED = 42
DATA = Path("../data")
PROC = DATA / "processed"
REF_DIR = DATA / "reference"
CTRL_DIR = DATA / "controle"
REF_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv("../.env")  # ANTHROPIC_API_KEY

True

## 1. L'échelle FRENCH

FRENCH est l'échelle de tri publiée par la Société française de médecine d'urgence (SFMU) pour les infirmiers d'accueil des urgences. C'est un tableau : pour chaque motif de recours, un niveau de tri par défaut, et des critères (constantes, signes associés) qui le font monter ou descendre. Les niveaux vont du tri 1 (le plus urgent) au tri 5, avec le tri 3 coupé en 3A et 3B.

On télécharge le référentiel (version 1.2, décembre 2019) et on en extrait le tableau, ligne par ligne.

In [2]:
FRENCH_URL = "https://www.sfmu.org/upload/referentielsSFMU/2019%2012%2006%20FRENCH%20pour%20PDF%20A4%20V1.2%20site.pdf"
pdf_path = REF_DIR / "french_v1.2.pdf"
if not pdf_path.exists():
    request = urllib.request.Request(FRENCH_URL, headers={"User-Agent": "Mozilla/5.0"})
    pdf_path.write_bytes(urllib.request.urlopen(request).read())

TRIS = ["1", "2", "3A", "3B", "4", "5"]


def cell_text(page, bbox):
    return re.sub(r"\s+", " ", page.get_textbox(pymupdf.Rect(bbox))).strip() if bbox else ""


lines = []
for page in pymupdf.open(pdf_path):
    for table in page.find_tables().tables:
        for row in table.rows:
            cells = [cell_text(page, b) for b in row.cells]
            if len(cells) < 8:
                continue
            motif, default, criteria = cells[0], cells[1], cells[2:8]
            if motif and not default and not any(criteria):  # titre de section (CARDIO-CIRCULATOIRE...)
                lines.append("## " + motif.title())
            elif default in TRIS:
                parts = [f"tri {t} : {c}" for t, c in zip(TRIS, criteria) if c]
                lines.append(f"- {motif} (tri par défaut : {default})" + "".join(" | " + p for p in parts))

FRENCH = "\n".join(lines)
print(sum(line.startswith("- ") for line in lines), "motifs de recours,", len(FRENCH), "caractères")
print("\n".join(lines[:4]))

115 motifs de recours, 15066 caractères
## Cardio-Circulatoire
- Arrêt cardiorespiratoire (tri par défaut : 1)
- Hypotension artérielle (tri par défaut : 2) | tri 1 : PAS ≤ 70 mmHg | tri 2 : PAS ≤ 90 mmHg ou PAS ≤ 100 mmHg et FC > 100/min | tri 3B : PAS 90 - 100 mmHg et FC ≤ 100/min
- Membre douloureux/ froid ou pâle/ischémie* (tri par défaut : 2) | tri 2 : durée ≤ 24 h ou cyanose/déficit moteur | tri 3B : durée ≥ 24 h


Regroupement dans les trois niveaux de la mission :
- urgence maximale : tri 1 et 2 (détresse vitale ou menace à court terme)
- urgence modérée : tri 3A et 3B
- urgence différée : tri 4 et 5

Le modèle donne le tri FRENCH, et c'est le code qui en déduit le niveau : le regroupement reste visible et identique pour tous les cas.

In [3]:
LEVELS = {"1": "urgence maximale", "2": "urgence maximale", "3A": "urgence modérée", "3B": "urgence modérée",
          "4": "urgence différée", "5": "urgence différée"}

## 2. Les cas cliniques

On reprend les cas cliniques des questions de MediQAl déjà dans le jeu. Les cas d'entraînement viennent de `sft_train`, les cas de test des jeux d'évaluation de MediQAl. MediQAl réutilise certains cas entre ses parties : un cas présent des deux côtés est écarté.

On prend d'abord les cas qui donnent des constantes (les plus proches d'un vrai triage), puis on complète au hasard. Après le tirage, on retire du test les cas presque identiques à un cas d'entraînement (même début de texte, à la ponctuation près) : la comparaison exacte en laissait passer.

In [4]:
def load(name):
    return [json.loads(line) for line in open(PROC / f"{name}.jsonl", encoding="utf-8")]


def clinical_cases(rows):
    # un même cas peut servir à plusieurs questions : on garde le premier exemple de chaque cas
    cases = {}
    for row in rows:
        text = row["messages"][0]["content"]
        if row["source"] == "mediqal" and text.startswith("Cas clinique :"):
            case = text.split("\n\nQuestion :")[0].removeprefix("Cas clinique :").strip()
            cases.setdefault(case, row)
    return cases


def has_vitals(row):
    return any(row["constantes"].values())


def draw(pool, n, rng):
    cases = sorted(pool)
    with_vitals = [c for c in cases if has_vitals(pool[c])]
    others = [c for c in cases if not has_vitals(pool[c])]
    rng.shuffle(with_vitals)
    rng.shuffle(others)
    return (with_vitals + others)[:n]


all_train = clinical_cases(load("sft_train"))
all_eval = clinical_cases(load("eval_mediqal_qcu") + load("eval_mediqal_qcm"))
shared = set(all_train) & set(all_eval)
train_pool = {c: r for c, r in all_train.items() if c not in shared}
test_pool = {c: r for c, r in all_eval.items() if c not in shared}
print(f"cas distincts : {len(all_train)} (entraînement), {len(all_eval)} (évaluation), {len(shared)} en commun écartés")

rng = random.Random(SEED)
cases = {"train": draw(train_pool, 400, rng), "test": draw(test_pool, 100, rng)}
rows = {**train_pool, **test_pool}


def key(case):
    # le même cas peut être recopié avec de petites différences : on compare le début du texte sans ponctuation
    return re.sub(r"[^a-z0-9]", "", case.lower())[:150]


train_keys = {key(c) for c in all_train}
near = [c for c in cases["test"] if key(c) in train_keys]
cases["test"] = [c for c in cases["test"] if key(c) not in train_keys]
print(len(near), "cas de test presque identiques à un cas d'entraînement, retirés")
for split, chosen in cases.items():
    print(f"{split} : {len(chosen)} cas, dont {sum(has_vitals(rows[c]) for c in chosen)} avec des constantes")

cas distincts : 666 (entraînement), 298 (évaluation), 85 en commun écartés
3 cas de test presque identiques à un cas d'entraînement, retirés
train : 400 cas, dont 145 avec des constantes
test : 97 cas, dont 49 avec des constantes


## 3. L'étiquetage par l'API

Le prompt système contient le rôle (infirmier d'accueil), le tableau FRENCH et la consigne. Il est identique pour tous les cas, donc mis en cache par l'API : il n'est facturé en entier qu'au premier appel.

La réponse est contrainte par un schéma (sortie structurée) : le motif retenu, le tri FRENCH, une explication et la conduite à tenir. Les réponses sont enregistrées dans `data/interim/triage_reponses.jsonl` : si on relance le notebook, l'API n'est rappelée que pour les cas manquants.

Le paramètre `fallbacks` fait reprendre la demande par un autre modèle si Opus 5.5 refuse de répondre. Le modèle qui a vraiment répondu est gardé dans les métadonnées.

In [5]:
SYSTEM = f'''Tu es infirmier organisateur de l'accueil (IOA) dans un service d'urgences en France. Tu attribues un niveau de tri avec l'échelle FRENCH de la SFMU (version 1.2, 2019), dont voici le tableau :

{FRENCH}

Pour chaque cas clinique :
- repère le motif de recours principal dans le tableau
- pars du tri par défaut de ce motif, puis applique les critères de gravité présents dans le cas (constantes, signes associés)
- décide d'après la situation à l'arrivée du patient, même si le cas raconte aussi la suite de la prise en charge
- n'invente pas d'information absente du cas
- réponds en français : une explication de deux ou trois phrases qui cite les éléments du cas utilisés, et la conduite à tenir à l'accueil en une phrase'''

MODEL = "claude-opus-5-5"
PRICES = {"input_tokens": 4.00, "cache_creation_input_tokens": 5.00, "cache_read_input_tokens": 0.20,
          "output_tokens": 20.00}  # dollars par million de tokens, Opus 5.5 (novembre 2026)


class Triage(BaseModel):
    motif: str
    tri: Literal["1", "2", "3A", "3B", "4", "5"]
    explication: str
    conduite: str


client = anthropic.Anthropic(max_retries=5)


def ask(case):
    response = client.beta.messages.parse(
        model=MODEL,
        max_tokens=8000,
        system=[{"type": "text", "text": SYSTEM, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": "Cas clinique : " + case}],
        output_format=Triage,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    usage = {k: getattr(response.usage, k) or 0 for k in PRICES}
    answer = response.parsed_output.model_dump() if response.stop_reason != "refusal" and response.parsed_output else None
    return {"cas": case, "modele": response.model, "stop_reason": response.stop_reason, "reponse": answer, "usage": usage}


def cost(usage):
    return sum(usage[k] * PRICES[k] for k in PRICES) / 1e6

Premier essai sur trois cas, pour voir les réponses et mesurer le coût avant de tout lancer.

In [6]:
answers_path = DATA / "interim" / "triage_reponses.jsonl"
answers = {}
if answers_path.exists():
    for line in open(answers_path, encoding="utf-8"):
        a = json.loads(line)
        answers[a["cas"]] = a

trial = [answers.get(c) or ask(c) for c in cases["train"][:3]]
for t in trial:
    print(t["cas"][:200], "...")
    print(" ->", t["reponse"])
    print("    ", t["modele"], t["usage"], f"{cost(t['usage']):.4f} $\n")
per_case = sum(cost(t["usage"]) for t in trial[1:]) / 2  # le premier appel paie l'écriture du cache
print(f"coût estimé pour 500 cas : {cost(trial[0]['usage']) + 499 * per_case:.2f} $")

Une petite fille naît à terme, après une grossesse sans problème, au poids de 3 350 g. Le premier mois de vie se déroule normalement. Elle est alimentée au biberon avec un lait maternisé. A l'âge de 6 ...
 -> {'motif': 'Dyspnée / insuffisance respiratoire (toux quinteuse cyanosante, suspicion de coqueluche) chez un nourrisson de 7 semaines', 'tri': '2', 'explication': "Ce nourrisson de moins de 3 mois a une toux répétitive et émétisante, avec des quintes cyanosantes depuis 48 heures et une polypnée à 65/min, ce qui fait suspecter une coqueluche à risque d'apnées et de forme maligne. Les signes respiratoires associés et les épisodes de cyanose dépassent le tri 3B par défaut et justifient un tri 2. Le tri 1 n'est pas retenu car le cas ne décrit pas de détresse respiratoire permanente à l'arrivée : l'état général est bon et la température à 37,8 °C.", 'conduite': "Installer immédiatement l'enfant en box en isolement respiratoire (gouttelettes), sous scope cardiorespiratoire avec SpO2 et m

In [7]:
for t in trial:
    answers.setdefault(t["cas"], t)

todo = [c for split in cases.values() for c in split if c not in answers]
print(len(todo), "cas à envoyer à l'API")
with ThreadPoolExecutor(max_workers=8) as pool:
    for result in pool.map(ask, todo):
        answers[result["cas"]] = result

with open(answers_path, "w", encoding="utf-8") as f:
    for a in answers.values():
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

done = [answers[c] for split in cases.values() for c in split]
print("réponses :", sum(a["reponse"] is not None for a in done), "sur", len(done))
print("modèles :", pd.Series([a["modele"] for a in done]).value_counts().to_dict())
print(f"coût total payé (tous les appels) : {sum(cost(a['usage']) for a in answers.values()):.2f} $")

0

 cas à envoyer à l'API
réponses : 497 sur 497
modèles : {'claude-opus-5-5': 497}
coût total payé (tous les appels) : 6.89 $


## 4. Contrôles

Répartition des niveaux, puis un contrôle automatique : quand une constante dépasse un seuil de tri 1 ou 2 du tableau FRENCH (PAS ≤ 90 mmHg, FC ≥ 130 ou ≤ 40/min, FR ≥ 30/min, SpO2 ≤ 90 %), le cas devrait être en urgence maximale. On regarde les cas qui ne le sont pas.

In [8]:
table = pd.DataFrame([{"split": split, "cas": c, **answers[c]["reponse"]}
                      for split, chosen in cases.items() for c in chosen if answers[c]["reponse"]])
table["niveau"] = table["tri"].map(LEVELS)
print(pd.crosstab(table["tri"], table["split"]), "\n")
print(pd.crosstab(table["niveau"], table["split"]))

split  test  train
tri               
1         5     25
2        38    151
3A        1     10
3B       38    132
4        10     39
5         5     43 

split             test  train
niveau                       
urgence différée    15     82
urgence maximale    43    176
urgence modérée     39    142


176 cas d'entraînement sur 400 sont en urgence maximale. Les cas d'examen décrivent souvent des situations graves, et le tirage a pris d'abord les cas avec des constantes : la répartition reflète ce choix, pas celle d'un vrai service d'urgences.

In [9]:
def number(value):
    match = re.match(r"\d+(?:[.,]\d+)?", value or "")
    return float(match.group().replace(",", ".")) if match else None


def vital_alert(vitals):
    pas, fc, fr, spo2 = (number(vitals[k]) for k in ("pa", "fc", "fr", "spo2"))
    return ((pas is not None and pas <= 90) or (fc is not None and (fc >= 130 or fc <= 40))
            or (fr is not None and fr >= 30) or (spo2 is not None and spo2 <= 90))


table["alerte_constantes"] = [vital_alert(rows[c]["constantes"]) for c in table["cas"]]
alerts = table[table["alerte_constantes"]]
print(len(alerts), "cas avec une constante au seuil de tri 1 ou 2,",
      (alerts["niveau"] != "urgence maximale").sum(), "classés autrement :")
for _, r in alerts[alerts["niveau"] != "urgence maximale"].iterrows():
    print(f"- tri {r['tri']} | constantes {rows[r['cas']]['constantes']} | {r['explication'][:220]}")

56 cas avec une constante au seuil de tri 1 ou 2, 7 classés autrement :
- tri 5 | constantes {'fc': '130', 'pa': None, 'fr': None, 'spo2': None, 'temperature': None} | Le motif « Fièvre ≤ 3 mois » ne s'applique pas à ce nourrisson de 10 mois ; on retient donc le motif « Fièvre », de tri 5 par défaut. La température est à 39,7 °C, donc sous le seuil de 40 °C. Il n'y a ni purpura, ni éru
- tri 3B | constantes {'fc': '24', 'pa': '110/60', 'fr': '24', 'spo2': None, 'temperature': '37 °C'} | Le motif retenu est « Ecchymose / hématome spontané(e) », tri par défaut 3B : éruption de pétéchies et d'ecchymoses généralisée, apparue progressivement en 3 jours. Il n'y a pas d'argument pour surclasser : apyrexie (37 
- tri 3B | constantes {'fc': '120', 'pa': None, 'fr': '48', 'spo2': None, 'temperature': '39.5 °C'} | Enfant de 4 ans avec toux et fièvre depuis 4 jours, T 39,5 °C et signes en foyer de la base droite (râles crépitants, submatité, augmentation des vibrations vocales) : la fièvre et les 

Les 7 cas relus un par un :
- 2 tensions écrites en centimètres de mercure dans la source (18/10 et 19/12, soit 180/100 et 190/120 mmHg). Le repérage du notebook 01 les lit comme une PAS de 18 ou 19, le modèle a bien lu 190/120
- un pouls à 24/min chez une fillette de 12 ans : la faute de frappe de la source déjà notée au notebook 01
- 4 enfants. Les seuils du tableau FRENCH sont pensés pour l'adulte : un pouls à 130/min chez un nourrisson de 10 mois ou une PA à 90/50 à 5 ans sont proches de la normale. Deux cas restent discutables, à regarder à la relecture : un enfant de 4 ans avec une FR à 48/min et un enfant de 8 ans avec un pouls à 130/min, tous deux en tri 3B

Ce contrôle repère surtout les limites du repérage des constantes. Il ne dit rien des cas sans constantes, d'où la relecture à la main.

Relecture à la main : 50 cas tirés au hasard (40 d'entraînement, 10 de test). Pour chacun, on note dans la colonne `d_accord` si le niveau paraît raisonnable (1) ou non (0), avec une remarque si besoin, et on enregistre le fichier sous `triage_relecture_annote.csv`. Il reste dans `data/controle/` et n'est pas publié.

In [10]:
review_path = CTRL_DIR / "triage_relecture.csv"
review = pd.concat([table[table["split"] == "train"].sample(40, random_state=SEED),
                    table[table["split"] == "test"].sample(10, random_state=SEED)])
if not review_path.exists():
    review[["split", "cas", "motif", "tri", "niveau", "explication", "conduite"]].assign(d_accord="", remarque="").to_csv(
        review_path, index=False, sep=";", encoding="utf-8-sig")  # s'ouvre directement dans Excel

annotated_path = CTRL_DIR / "triage_relecture_annote.csv"
if annotated_path.exists():
    annotated = pd.read_csv(annotated_path, sep=";", encoding="utf-8-sig")
    print(f"relecture : {int(annotated['d_accord'].sum())} niveaux jugés raisonnables sur {len(annotated)}")
else:
    print(review_path.name, ": 50 cas à relire")

relecture : 50 niveaux jugés raisonnables sur 50


## 5. Le jeu de triage

Chaque cas devient une paire question-réponse au même format que le jeu SFT. La réponse donne le niveau de la mission, le tri FRENCH, l'explication et la conduite à tenir. Les métadonnées indiquent l'origine de l'étiquette et une confiance faible.

In [11]:
QUESTION = "Quel est le niveau de priorité de ce patient à l'accueil des urgences ?"


def to_example(case):
    row, a = rows[case], answers[case]
    level = LEVELS[a["reponse"]["tri"]]
    reply = (f"Niveau : {level} (tri {a['reponse']['tri']} de l'échelle FRENCH).\n"
             f"Explication : {a['reponse']['explication']}\n"
             f"Conduite : {a['reponse']['conduite']}")
    return {"id": "triage-" + row["id"],
            "messages": [{"role": "user", "content": f"Cas clinique : {case}\n\n{QUESTION}"},
                         {"role": "assistant", "content": reply}],
            "source": "mediqal", "sous_source": row["sous_source"], "id_origine": row["id_origine"],
            "licence": row["licence"], "langue": "fr", "tache": "triage", "specialite": row["specialite"],
            "type_question": None, "origine_annotation": f"{a['modele']} (API Anthropic), échelle FRENCH de la SFMU",
            "niveau_confiance": "faible", "symptomes": row["symptomes"], "antecedents": row["antecedents"],
            "constantes": row["constantes"], "version": "1.1", "nb_elements_masques": case.count("<PERSONNE>"),
            "tri_french": a["reponse"]["tri"], "niveau": level}


for split, chosen in cases.items():
    examples = [to_example(c) for c in chosen if answers[c]["reponse"]]
    with open(PROC / f"triage_{split}.jsonl", "w", encoding="utf-8") as f:
        for e in examples:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"triage_{split}.jsonl : {len(examples)} exemples")

print("\n" + examples[0]["messages"][0]["content"][:300], "...\n")
print(examples[0]["messages"][1]["content"])

triage_train.jsonl : 400 exemples
triage_test.jsonl : 97 exemples

Cas clinique : un nourrisson de 08 mois est amené par ses parents en consultation en raison
de mouvements anormaux. La mère a constaté ¡'apparition brutale le matin d'un
raidissement de l'enfant suivi de secousses des membres. L'épisode a été bref.
C'est un nourrisson né prématurément à 32 semaines  ...

Niveau : urgence modérée (tri 3A de l'échelle FRENCH).
Explication : Nourrisson de 8 mois ayant présenté brutalement un raidissement suivi de secousses des membres, épisode bref, dans un contexte fébrile (T° 38,5 °C, rhinopharyngite). L'examen clinique à l'arrivée est normal, ce qui traduit une récupération complète, sans notion de récidive, de durée ≥ 10 min ni d'hypotonie : le tri par défaut 3B est abaissé en 3A. Les antécédents (prématurité à 32 SA, souffrance néonatale, antécédents familiaux) sont à transmettre au médecin mais ne modifient pas le niveau de tri selon les critères FRENCH.
Conduite : Installer l'enfant

## 6. Publication (version 1.1)

Les fichiers de triage rejoignent le jeu publié sur Hugging Face, avec la fiche mise à jour et l'étiquette `v1.1`. On ne publie qu'une fois la relecture faite.

In [12]:
if annotated_path.exists():
    from huggingface_hub import HfApi

    api = HfApi()
    repo_id = f"{api.whoami()['name']}/medical-triage-poc-data"
    api.upload_folder(folder_path=PROC, repo_id=repo_id, repo_type="dataset", allow_patterns=["triage_*.jsonl"],
                      commit_message="Cas de triage, version 1.1")
    api.upload_file(path_or_fileobj=Path("../dataset/README.md"), path_in_repo="README.md", repo_id=repo_id,
                    repo_type="dataset", commit_message="Fiche du jeu de données, version 1.1")
    api.create_tag(repo_id, tag="v1.1", repo_type="dataset", exist_ok=True)
    print("publié :", f"https://huggingface.co/datasets/{repo_id}")
else:
    print("publication après la relecture des 50 cas")

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


publié : https://huggingface.co/datasets/g-hostin/medical-triage-poc-data
